# Batchnorm 2d

Сверточный слой можно свести к линейному слою. BatchNorm можно применять и для линейных слоев, и для сверточных.

Признаки внутри одного канала (одной карты признаков) получаются путем применения к исходному изображению одних и тех же преобразований. По сути **1 канал** — это **карта одного признака**. Поэтому логично, что нормализация будет происходить по каналам, а одним признаком считается одна **feature map**.
Нормализация идет по всей такой feature map (по всему каналу) для всех объектов батча.

<center><img src ="https://ml.gan4x4.ru/msu/dev-2.2/L07/out/feature_map.png" width="500"></center>

В PyTorch для полносвязных слоев используют `nn.BatchNorm1d` [🛠️[doc]](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm1d.html), а для сверточных — `nn.BatchNorm2d` [🛠️[doc]](https://pytorch.org/docs/stable/generated/torch.nn.BatchNorm2d.html).

Графически нормализации часто объясняют при помощи следующей схемы:

<center><img src ="https://ml.gan4x4.ru/msu/dev-2.2/L07/out/dimensions_channels_batch_samples.png" width="450"></center>

*По оси абсцисс* расположены **объекты из батча**,
*по оси ординат* &mdash; **feature map**, преобразованный в вектор,
*по оси аппликат* &mdash; **каналы** (feature maps).

Размер такого кубика:
$$[\text{batch size}, \text{channels}, \text{h}, \text{w}]$$

В этом представлении **BatchNorm** для свертки выглядит следующим образом:

<center><img src ="https://ml.gan4x4.ru/msu/dev-2.2/L07/out/visualization_of_batch_normalization.png" width="450">

<em>Source: <a href="https://paperswithcode.com/method/batch-normalization">Batch Normalization</a></em></center>

Статистки считаются по срезу:

$$[\text{:}, \text{ channels}, \text{ :}, \text{ :}]$$

Всего считается $\text{channels}$ статистик, по одной для каждой карты признаков.

## Другие Normalization

Существует большое количество иных нормализаций, их неполный список можно найти [здесь 🎓[article]](https://paperswithcode.com/methods/category/normalization). В данной секции мы рассмотрим самые популярные виды нормализации помимо **BatchNorm**.


Есть несколько мотиваций для замены батч-нормализации на другие типы нормализации:
- при обучении модели вы можете столкнуться с ситуацией, когда вам нужно учить модель при `batch_size=1` (больше не помещается на видеокарту). BN работает неэффективно с маленькими размерами батча,
- при параллельном обучении модели на нескольких видеокартах необходимо обмениваться статистиками для BN между устройствами. Обмен информацией между устройствами может замедлять обучение.

<center><img src ="https://ml.gan4x4.ru/msu/dev-2.1/L07/normalization_methods.png" width="900">

<em>Source: <a href="https://paperswithcode.com/method/layer-normalization">Layer Normalization</a></em></center>

### Layer Norm

Помимо свёрточных нейронных сетей, существует специальный тип нейронных сетей, используемых для обработки последовательностей. Называется он "**рекуррентные нейронные сети**".

Когда оказалось, что **BatchNorm** положительно сказывается на обучении нейронных сетей, его попытались применить для различных архитектур. BatchNorm нельзя было использовать "из коробки" для рекуррентных нейронных сетей (работающих с последовательными данными), пришлось придумывать различные адаптации, среди которых наиболее удачной оказалась **Layer Normalization**.

По сути, теперь нормализация происходит внутри **одного объекта из батча**, а не поканально в рамках батча. С математической точки зрения данная "адаптация" отличается от **BatchNorm**, однако экспериментально она превзошла своих конкурентов в задаче нормализации при обработке последовательных данных.

Впоследствии данный метод нормализации хорошо проявил себя в **трансформерах** — наследниках рекуррентных нейронных сетей в вопросах обработки последовательных данных. После успешного применения трансформеров в задачах компьютерного зрения, **LayerNorm** стал использоваться и в компьютерном зрении.

Размер кубика:
$$[\text{batch size}, \text{channels}, \text{h}, \text{w}]$$

Статистики считаются по срезу:
$$[\text{batch size}, \text{ :}, \text{ :}, \text{ :}]$$

По одной статистике для кажого объекта в батче.

<center><img src ="https://ml.gan4x4.ru/msu/dev-2.2/L07/out/visualization_of_layer_normalization.png" width="450">

<em>Source: <a href="https://paperswithcode.com/method/layer-normalization">Layer Normalization</a></em></center>


### Instance Norm

Следующий вид нормализации был предложен отечественными исследователями (из Сколтеха), занимавшимися разработкой быстрого и эффективного способа **переноса стиля** с одного изображения на другое.

При использовании BatchNorm теряется информация о контрастах на конкретном изображении, поскольку нормализация производится по нескольким объектам. Для сохранения *контрастов* в экземпляре (*instance*) изображения была предложена специальная нормализация, рассматривающая конкретный канал одного конкретного объекта. Было предложено два названия нормализации: связанное с мотивацией (contrast normalization) и связанное с принципом работы (**instance normalization**).

Размер кубика:
$$[\text{batch size}, \text{channels}, \text{h}, \text{w}]$$

Статистики считаются по срезу:
$$[\text{batch size}, \text{channels}, \text{ :}, \text{ :}]$$

По одной статистике для каждого канала каждого объекта в батче.

<center><img src ="https://ml.gan4x4.ru/msu/dev-2.2/L07/out/visualization_of_instance_normalization.png" width="450"></center>

<center><em>Source: <a href="https://paperswithcode.com/method/instance-normalization">Instance Normalization</a></em></center>

### Group Norm

В течение долгого времени BatchNorm оставался однозначным фаворитом для использования в задачах компьютерного зрения, однако:
1. В связи с необходимостью точно считать статистики внутри batch при обучении приходилось использовать большой batch size.

2. Ограниченность размера памяти видеокарты вынуждает разработчиков идти на компромисс между сложностью модели и размером батча.

Таким образом, использование BatchNorm приводило к невозможности использовать сложные модели**\***, поскольку им просто не хватало места на видеокарте.

Необходимость использовать большой batch size могут решать различные нормализации, не использующие batch-размерность. К примеру, уже известные нам **Layer Norm** и **Instance Norm**. Эмпирически оказалось, что данные нормализации уступают BatchNorm по качеству работы: в то время как LayerNorm предполагает одинаковую важность и суть различных каналов (*рассматривая данные излишне глобально*), InstanceNorm упускает межканальные взаимодействия (*рассматривая данные слишком локально*).

Успешным обобщением данных методов является **Group Normalization**: данный метод разбивает каналы на $G \in [1; C]$ групп, присваивая каждой из них примерно равное количество каналов. Отметим, что при $G = 1$ GroupNorm идентичен LayerNorm, а при $G = C$ GroupNorm идентичен InstanceNorm.

Эмпирически оказалось, что при замене BatchNorm на GroupNorm качество модели падает в разы менее значительно, чем при использовании LayerNorm либо InstanceNorm. Более того, при изменении batch size качество работы GroupNorm не изменялось, что открыло перспективы для создания более сложных моделей компьютерного зрения.

**\*** — подразумевается, что уменьшение batch size позволило бы создать более сложные модели.

Размер кубика:
$$[\text{batch size}, \text{channels}, \text{h}, \text{w}]$$

Статистики считаются по срезу:
$$[\text{batch size}, \text{ i: j}, \text{ :}, \text{ :}]$$

По одной статистике для каждой группы каналов каждого объекта в батче.

<center><img src ="https://ml.gan4x4.ru/msu/dev-2.2/L07/out/visualization_of_group_normalization.png" width="450"></center>

<center><em>Source: <a href="https://paperswithcode.com/method/group-normalization">Group Normalization</a></em></center>

# Dropout 2d

Аналогично **Batch Normalization**, при применении к **сверточному слою**  **Dropout** должен **убирать каналы целиком**. Dropout для полносвязного слоя реализован в PyTorch в `nn.Dropout` [🛠️[doc]](https://pytorch.org/docs/stable/generated/torch.nn.Dropout.html), для сверточного — в `nn.Dropout2d` [🛠️[doc]](https://pytorch.org/docs/stable/generated/torch.nn.Dropout2d.html).

<center><img src ="https://ml.gan4x4.ru/msu/dev-2.2/L07/out/dropout_2d.png" width="700"></center>